# Comprehensive RAG Evaluation and Analysis

This notebook provides comprehensive evaluation of two RAG baselines:
1. **Normal RAG**: Standard retrieve-and-generate
2. **RAG + LLM Filter**: Two-stage filtering (context + answer)

Evaluation includes:
- **Filter Effectiveness**: % wrong answers filtered, % correct retained
- **Quality Metrics**: RAGAS metrics (faithfulness, relevancy, etc.)
- **Traditional NLG Metrics**: ROUGE-L, BERTScore
- **Error Analysis**: Categorization and case studies

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Setup complete")

## 2. Load Results

In [ ]:
# Load prediction results
results_dir = Path("../results")

df_normal = pd.read_csv(results_dir / "asqa_normal_rag_predictions.csv")
df_filtered = pd.read_csv(results_dir / "asqa_filtered_rag_predictions.csv")

print(f"Normal RAG predictions: {len(df_normal)}")
print(f"Filtered RAG predictions: {len(df_filtered)}")

# Load synthetic labeled data (ground truth for filter evaluation)
df_labeled = pd.read_csv("../data/asqa/synthetic_labeled_train.csv")
print(f"\nSynthetic labeled samples: {len(df_labeled)}")

## 3. Filter Effectiveness Metrics

### 3.1 Context Filter Analysis

In [ ]:
# Context filtering statistics
print("Context Filter Statistics:\n")

total_contexts = df_filtered['num_contexts'].sum()
filtered_contexts = df_filtered['num_filtered_contexts'].sum()
removed_contexts = total_contexts - filtered_contexts

filter_rate = (removed_contexts / total_contexts * 100) if total_contexts > 0 else 0

print(f"Total contexts retrieved: {total_contexts}")
print(f"Contexts after filtering: {filtered_contexts}")
print(f"Contexts removed: {removed_contexts}")
print(f"Filter rate: {filter_rate:.1f}%")

# Per-question statistics
avg_contexts_before = df_filtered['num_contexts'].mean()
avg_contexts_after = df_filtered['num_filtered_contexts'].mean()

print(f"\nAverage contexts per question:")
print(f"  Before filtering: {avg_contexts_before:.2f}")
print(f"  After filtering: {avg_contexts_after:.2f}")
print(f"  Reduction: {(avg_contexts_before - avg_contexts_after):.2f} ({filter_rate:.1f}%)")

### 3.2 Answer Filter Analysis

In [ ]:
# Answer quality distribution
print("Answer Filter Statistics:\n")

quality_counts = df_filtered['answer_quality'].value_counts()
total_answers = len(df_filtered)

good_count = quality_counts.get('GOOD', 0)
bad_count = quality_counts.get('BAD', 0)

print(f"Total answers evaluated: {total_answers}")
print(f"GOOD answers: {good_count} ({good_count/total_answers*100:.1f}%)")
print(f"BAD answers: {bad_count} ({bad_count/total_answers*100:.1f}%)")

# Score statistics
print(f"\nAverage Scores:")
print(f"  Faithfulness: {df_filtered['faithfulness_score'].mean():.2f}")
print(f"  Relevance: {df_filtered['relevance_score'].mean():.2f}")
print(f"  Completeness: {df_filtered['completeness_score'].mean():.2f}")

### 3.3 Filter Effectiveness with Ground Truth

In [ ]:
# Match filtered results with labeled data for ground truth evaluation
# This requires matching questions between datasets

# Create a simple matching based on question text
labeled_dict = {row['question']: row['label'] for _, row in df_labeled.iterrows()}

# Add ground truth labels to filtered results
df_filtered['ground_truth_label'] = df_filtered['question'].map(labeled_dict)

# Filter to only samples with ground truth
df_with_gt = df_filtered[df_filtered['ground_truth_label'].notna()].copy()

print(f"Samples with ground truth: {len(df_with_gt)}")

if len(df_with_gt) > 0:
    # Convert answer quality to binary prediction
    df_with_gt['predicted_label'] = (df_with_gt['answer_quality'] == 'GOOD').astype(int)
    
    # Calculate metrics
    y_true = df_with_gt['ground_truth_label'].astype(int)
    y_pred = df_with_gt['predicted_label']
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    print("\nFilter Performance (with ground truth):")
    print("\nConfusion Matrix:")
    print("                Predicted BAD  Predicted GOOD")
    print(f"Actual BAD      {cm[0,0]:13}  {cm[0,1]:14}")
    print(f"Actual GOOD     {cm[1,0]:13}  {cm[1,1]:14}")
    
    # Calculate key metrics
    tn, fp, fn, tp = cm.ravel()
    
    wrong_filtered = tn / (tn + fp) * 100 if (tn + fp) > 0 else 0  # Precision for BAD
    correct_retained = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0  # Recall for GOOD
    
    print(f"\nKey Metrics:")
    print(f"  Wrong answers filtered: {wrong_filtered:.1f}%")
    print(f"  Correct answers retained: {correct_retained:.1f}%")
    
    # Classification report
    print("\nDetailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['BAD', 'GOOD']))
else:
    print("\nWarning: No matching samples with ground truth found.")
    print("Filter effectiveness metrics require ground truth labels.")

## 4. Quality Metrics with RAGAS

In [ ]:
# Import RAGAS evaluator
from src.evaluation.ragas_evaluator import compare_rag_systems

print("Running RAGAS evaluation...")
print("Note: This may take several minutes")

try:
    # Compare systems
    comparison_df = compare_rag_systems(
        df_normal,
        df_filtered,
        system1_name="Normal RAG",
        system2_name="Filtered RAG",
        metrics=['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
    )
    
    print("\n" + "="*80)
    print("RAGAS Metrics Comparison")
    print("="*80)
    print(comparison_df.to_string(index=False))
    
    # Save comparison
    comparison_df.to_csv(results_dir / "ragas_comparison.csv", index=False)
    print("\n✅ Saved RAGAS comparison to results/ragas_comparison.csv")
    
except Exception as e:
    print(f"\nError running RAGAS evaluation: {e}")
    print("This may be due to missing API keys or RAGAS installation issues.")
    print("Continuing with other metrics...")

## 5. Traditional NLG Metrics

In [ ]:
# Install if needed
try:
    from rouge_score import rouge_scorer
    from bert_score import score as bert_score
except ImportError:
    print("Installing rouge-score and bert-score...")
    !pip install -q rouge-score bert-score
    from rouge_score import rouge_scorer
    from bert_score import score as bert_score

print("✅ NLG metrics libraries loaded")

In [ ]:
# Calculate ROUGE-L scores
print("Calculating ROUGE-L scores...")

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def calculate_rouge_l(predictions, references):
    scores = []
    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)
        scores.append(score['rougeL'].fmeasure)
    return np.mean(scores)

rouge_normal = calculate_rouge_l(
    df_normal['predicted_answer'].tolist(),
    df_normal['gold_answer'].tolist()
)

rouge_filtered = calculate_rouge_l(
    df_filtered['predicted_answer'].tolist(),
    df_filtered['gold_answer'].tolist()
)

print(f"\nROUGE-L Scores:")
print(f"  Normal RAG: {rouge_normal:.4f}")
print(f"  Filtered RAG: {rouge_filtered:.4f}")
print(f"  Improvement: {(rouge_filtered - rouge_normal):.4f} ({(rouge_filtered - rouge_normal)/rouge_normal*100:+.1f}%)")

In [ ]:
# Calculate BERTScore (sample for speed)
print("Calculating BERTScore (on 100 samples)...")

sample_size = min(100, len(df_normal))

# Normal RAG
P_normal, R_normal, F1_normal = bert_score(
    df_normal['predicted_answer'].head(sample_size).tolist(),
    df_normal['gold_answer'].head(sample_size).tolist(),
    lang='en',
    verbose=False
)

# Filtered RAG
P_filtered, R_filtered, F1_filtered = bert_score(
    df_filtered['predicted_answer'].head(sample_size).tolist(),
    df_filtered['gold_answer'].head(sample_size).tolist(),
    lang='en',
    verbose=False
)

print(f"\nBERTScore F1 (sample of {sample_size}):")
print(f"  Normal RAG: {F1_normal.mean():.4f}")
print(f"  Filtered RAG: {F1_filtered.mean():.4f}")
print(f"  Improvement: {(F1_filtered.mean() - F1_normal.mean()):.4f}")

## 6. Comprehensive Comparison Table

In [ ]:
# Create comprehensive comparison table
comparison_data = [
    {
        'Category': 'Filter Effectiveness',
        'Metric': 'Context Filter Rate',
        'Normal RAG': 'N/A',
        'Filtered RAG': f"{filter_rate:.1f}%",
        'Δ': 'N/A'
    },
    {
        'Category': 'Filter Effectiveness',
        'Metric': 'Answer Filter Rate (BAD)',
        'Normal RAG': 'N/A',
        'Filtered RAG': f"{bad_count/total_answers*100:.1f}%",
        'Δ': 'N/A'
    },
    {
        'Category': 'Traditional NLG',
        'Metric': 'ROUGE-L',
        'Normal RAG': f"{rouge_normal:.4f}",
        'Filtered RAG': f"{rouge_filtered:.4f}",
        'Δ': f"{(rouge_filtered - rouge_normal):+.4f}"
    },
    {
        'Category': 'Traditional NLG',
        'Metric': 'BERTScore F1',
        'Normal RAG': f"{F1_normal.mean():.4f}",
        'Filtered RAG': f"{F1_filtered.mean():.4f}",
        'Δ': f"{(F1_filtered.mean() - F1_normal.mean()):+.4f}"
    }
]

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "="*100)
print("COMPREHENSIVE EVALUATION COMPARISON")
print("="*100)
print(df_comparison.to_string(index=False))

# Save comparison
df_comparison.to_csv(results_dir / "evaluation_report.csv", index=False)
print("\n✅ Saved evaluation report to results/evaluation_report.csv")

## 7. Visualizations

In [ ]:
# Create comprehensive visualization
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Context filtering
ax1 = fig.add_subplot(gs[0, 0])
contexts_data = [avg_contexts_before, avg_contexts_after]
ax1.bar(['Before Filter', 'After Filter'], contexts_data, color=['#3498db', '#2ecc71'])
ax1.set_ylabel('Avg Contexts per Question')
ax1.set_title('Context Filtering Effect')
ax1.grid(axis='y', alpha=0.3)

# 2. Answer quality distribution
ax2 = fig.add_subplot(gs[0, 1])
quality_data = [good_count, bad_count]
colors = ['#2ecc71', '#e74c3c']
ax2.pie(quality_data, labels=['GOOD', 'BAD'], autopct='%1.1f%%', colors=colors, startangle=90)
ax2.set_title('Answer Quality Distribution')

# 3. Score distributions
ax3 = fig.add_subplot(gs[0, 2])
scores = ['Faithfulness', 'Relevance', 'Completeness']
score_values = [
    df_filtered['faithfulness_score'].mean(),
    df_filtered['relevance_score'].mean(),
    df_filtered['completeness_score'].mean()
]
ax3.barh(scores, score_values, color='#9b59b6')
ax3.set_xlabel('Average Score')
ax3.set_title('LLM Judge Scores')
ax3.set_xlim(0, 10)
ax3.grid(axis='x', alpha=0.3)

# 4. Confusion matrix (if available)
if len(df_with_gt) > 0:
    ax4 = fig.add_subplot(gs[1, 0])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4,
                xticklabels=['Pred BAD', 'Pred GOOD'],
                yticklabels=['True BAD', 'True GOOD'])
    ax4.set_title('Filter Confusion Matrix')
    ax4.set_ylabel('True Label')
    ax4.set_xlabel('Predicted Label')

# 5. ROUGE-L comparison
ax5 = fig.add_subplot(gs[1, 1])
systems = ['Normal RAG', 'Filtered RAG']
rouge_scores = [rouge_normal, rouge_filtered]
bars = ax5.bar(systems, rouge_scores, color=['#3498db', '#2ecc71'])
ax5.set_ylabel('ROUGE-L F1')
ax5.set_title('ROUGE-L Comparison')
ax5.set_ylim(0, max(rouge_scores) * 1.2)
ax5.grid(axis='y', alpha=0.3)
# Add value labels
for bar in bars:
    height = bar.get_height()
    ax5.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom')

# 6. BERTScore comparison
ax6 = fig.add_subplot(gs[1, 2])
bert_scores = [F1_normal.mean().item(), F1_filtered.mean().item()]
bars = ax6.bar(systems, bert_scores, color=['#3498db', '#2ecc71'])
ax6.set_ylabel('BERTScore F1')
ax6.set_title('BERTScore Comparison')
ax6.set_ylim(0, max(bert_scores) * 1.2)
ax6.grid(axis='y', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax6.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom')

# 7. Score distribution histograms
ax7 = fig.add_subplot(gs[2, :])
ax7.hist(df_filtered['faithfulness_score'], bins=20, alpha=0.5, label='Faithfulness', color='#3498db')
ax7.hist(df_filtered['relevance_score'], bins=20, alpha=0.5, label='Relevance', color='#2ecc71')
ax7.hist(df_filtered['completeness_score'], bins=20, alpha=0.5, label='Completeness', color='#9b59b6')
ax7.set_xlabel('Score')
ax7.set_ylabel('Frequency')
ax7.set_title('Score Distributions (Filtered RAG)')
ax7.legend()
ax7.grid(axis='y', alpha=0.3)

plt.suptitle('Comprehensive RAG Evaluation Analysis', fontsize=16, fontweight='bold', y=0.995)
plt.savefig(results_dir / 'comprehensive_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Saved visualization to results/comprehensive_evaluation.png")

## 8. Error Analysis

In [ ]:
# Analyze filtered BAD answers
bad_answers = df_filtered[df_filtered['answer_quality'] == 'BAD'].copy()

print(f"Error Analysis: {len(bad_answers)} BAD answers\n")

# Categorize by score patterns
bad_answers['error_type'] = 'Unknown'

# Low faithfulness = hallucination
bad_answers.loc[bad_answers['faithfulness_score'] < 5, 'error_type'] = 'Hallucination'

# Low relevance = irrelevant
bad_answers.loc[(bad_answers['relevance_score'] < 5) & 
                (bad_answers['faithfulness_score'] >= 5), 'error_type'] = 'Irrelevant'

# Low completeness = incomplete
bad_answers.loc[(bad_answers['completeness_score'] < 5) & 
                (bad_answers['faithfulness_score'] >= 5) &
                (bad_answers['relevance_score'] >= 5), 'error_type'] = 'Incomplete'

error_counts = bad_answers['error_type'].value_counts()
print("Error Type Distribution:")
for error_type, count in error_counts.items():
    print(f"  {error_type}: {count} ({count/len(bad_answers)*100:.1f}%)")

In [ ]:
# Show examples of each error type
print("\nError Examples:\n")

for error_type in error_counts.index[:3]:  # Top 3 error types
    examples = bad_answers[bad_answers['error_type'] == error_type].head(1)
    
    for _, row in examples.iterrows():
        print(f"{'='*80}")
        print(f"Error Type: {error_type}")
        print(f"{'='*80}")
        print(f"Question: {row['question']}")
        print(f"\nAnswer: {row['predicted_answer'][:300]}...")
        print(f"\nScores: F={row['faithfulness_score']:.1f}, R={row['relevance_score']:.1f}, C={row['completeness_score']:.1f}")
        print(f"Reasoning: {row['filter_reasoning']}")
        print()

## 9. Summary Report

In [ ]:
# Generate summary report
summary = f"""
{'='*80}
ASQA RAG EVALUATION SUMMARY REPORT
{'='*80}

1. DATASET
   - Training samples: {len(df_normal)}
   - Dev samples: {len(df_filtered)}
   - Synthetic labeled samples: {len(df_labeled)}

2. FILTER EFFECTIVENESS
   Context Filtering:
   - Total contexts retrieved: {total_contexts}
   - Filter rate: {filter_rate:.1f}%
   - Avg contexts before: {avg_contexts_before:.2f}
   - Avg contexts after: {avg_contexts_after:.2f}
   
   Answer Filtering:
   - GOOD answers: {good_count} ({good_count/total_answers*100:.1f}%)
   - BAD answers: {bad_count} ({bad_count/total_answers*100:.1f}%)
   - Avg faithfulness: {df_filtered['faithfulness_score'].mean():.2f}
   - Avg relevance: {df_filtered['relevance_score'].mean():.2f}
   - Avg completeness: {df_filtered['completeness_score'].mean():.2f}

3. QUALITY METRICS
   ROUGE-L:
   - Normal RAG: {rouge_normal:.4f}
   - Filtered RAG: {rouge_filtered:.4f}
   - Improvement: {(rouge_filtered - rouge_normal):+.4f} ({(rouge_filtered - rouge_normal)/rouge_normal*100:+.1f}%)
   
   BERTScore F1:
   - Normal RAG: {F1_normal.mean():.4f}
   - Filtered RAG: {F1_filtered.mean():.4f}
   - Improvement: {(F1_filtered.mean() - F1_normal.mean()):+.4f}

4. ERROR ANALYSIS
   Top error types in filtered BAD answers:
"""

for error_type, count in error_counts.items():
    summary += f"   - {error_type}: {count} ({count/len(bad_answers)*100:.1f}%)\n"

summary += f"""
5. KEY FINDINGS
   - LLM filtering removed {filter_rate:.1f}% of retrieved contexts
   - {bad_count/total_answers*100:.1f}% of generated answers were flagged as BAD
   - ROUGE-L {'improved' if rouge_filtered > rouge_normal else 'decreased'} by {abs((rouge_filtered - rouge_normal)/rouge_normal*100):.1f}%
   - Most common error type: {error_counts.index[0] if len(error_counts) > 0 else 'N/A'}

6. RECOMMENDATIONS
   - Consider adjusting filter thresholds based on precision/recall trade-off
   - Focus on improving {error_counts.index[0] if len(error_counts) > 0 else 'N/A'} errors
   - Evaluate filter impact on different question types
   - Consider ensemble filtering with multiple judges

{'='*80}
"""

print(summary)

# Save summary
with open(results_dir / 'evaluation_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary)

print("\n✅ Saved summary report to results/evaluation_summary.txt")

## Summary

Comprehensive evaluation completed:

✅ **Filter Effectiveness Metrics**
- Context filter rate calculated
- Answer quality distribution analyzed
- Ground truth comparison (when available)

✅ **Quality Metrics**
- RAGAS metrics (faithfulness, relevancy, etc.)
- ROUGE-L scores
- BERTScore F1

✅ **Analysis**
- Comprehensive comparison table
- Visualizations
- Error categorization
- Summary report

All results saved to `results/` directory.